> **Unit 511 · 511-101 Introduction to AI Systems · Lesson 2**
> "Tokens and Embeddings" by Unit 511 (Fairytale.ai and Unit 511 contributors), https://unit511.fairytale.ai/curriculum/511-101/02-tokens-and-embeddings/, licensed CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Code in this notebook is MIT licensed.

# Tokens and Embeddings

This notebook accompanies [lesson 2](https://unit511.fairytale.ai/curriculum/511-101/02-tokens-and-embeddings/). Run each cell in order. Everything uses the Python standard library, so it runs in Colab or any Python 3.10+ environment with nothing to install.

## 1. Words vs. characters

In [ ]:
text = "Tokenizers split unbelievably long words."
print(text.split())
print(list("unbelievably"))

## 2. Byte-pair encoding

Start from characters, then repeatedly merge the most frequent adjacent pair. `_` marks the end of a word.

In [ ]:
from collections import Counter


def most_common_pair(words):
    """Count every adjacent pair of symbols, weighted by word frequency."""
    pairs = Counter()
    for word, freq in words.items():
        for a, b in zip(word, word[1:]):
            pairs[(a, b)] += freq
    return pairs.most_common(1)[0][0]


def merge(words, pair):
    """Replace every occurrence of `pair` with a single merged symbol."""
    merged = {}
    for word, freq in words.items():
        out, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                out.append(word[i] + word[i + 1])
                i += 2
            else:
                out.append(word[i])
                i += 1
        merged[tuple(out)] = freq
    return merged


corpus = "low low low lower lowest newer newer newest wider widest".split()
words = Counter(tuple(w) + ("_",) for w in corpus)  # "_" marks the end of a word
merges = []
for step in range(8):
    pair = most_common_pair(words)
    merges.append(pair)
    words = merge(words, pair)
    print(f"merge {step + 1}: {pair[0]!r} + {pair[1]!r}")

for word in words:
    print(" ".join(word))

## 3. Encoding new words

Apply the learned merges in order, then map each token to an integer ID.

In [ ]:
def encode(word, merges):
    symbols = list(word) + ["_"]
    for pair in merges:
        symbols = list(merge({tuple(symbols): 1}, pair))[0]
    return list(symbols)

vocab = sorted({s for w in words for s in w} | set("abcdefghijklmnopqrstuvwxyz_"))
ids = {s: i for i, s in enumerate(vocab)}

for word in ["lowest", "newest", "slowest"]:
    tokens = encode(word, merges)
    print(word, "->", tokens, "->", [ids[t] for t in tokens])

## 4. Embeddings and cosine similarity

These 4-dimensional vectors are hand-made toys. Real embeddings are learned and have hundreds or thousands of dimensions.

In [ ]:
import math

embeddings = {
    "cat":    [0.90, 0.80, 0.10, 0.00],
    "kitten": [0.85, 0.75, 0.20, 0.05],
    "dog":    [0.80, 0.30, 0.10, 0.05],
    "pizza":  [0.05, 0.10, 0.90, 0.80],
}


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b)


for word in ["kitten", "dog", "pizza"]:
    print(f"cat vs {word:<6} {cosine(embeddings['cat'], embeddings[word]):.3f}")

## 5. Nearest neighbours

In [ ]:
def most_similar(query, embeddings, k=2):
    scores = [
        (word, round(cosine(embeddings[query], vector), 3))
        for word, vector in embeddings.items()
        if word != query
    ]
    return sorted(scores, key=lambda pair: pair[1], reverse=True)[:k]

print(most_similar("pizza", embeddings))
print(most_similar("kitten", embeddings, k=3))

## Exercise: grow your tokenizer

1. Change the BPE loop in section 2 to run 12 merges. Which new tokens appear? Does `newest` encode into fewer pieces?
2. Write `decode(tokens)` below and check that `decode(encode(w, merges)) == w` for every word in the corpus and for `"slowest"`.
3. In the markdown cell at the end, explain in two sentences why `encode` must apply merges in order.

In [ ]:
def decode(tokens):
    # Your code here
    ...


for w in ["low", "lower", "lowest", "newer", "newest", "wider", "widest", "slowest"]:
    print(w, decode(encode(w, merges)) == w)

**Why merges must be applied in order:** _write your explanation here._